# Actividad 15v4 — TCN como Modelo Competidor
**Autor:** Fabrizio Sanchez Saravia — UPeU Juliaca

## Justificacion
TCN (Temporal Convolutional Network) es un competidor justo para Small Data porque:
- Usa convoluciones dilatadas en vez de recurrencia -> menos parametros que LSTM
- Captura patrones de corto y largo plazo simultaneamente
- Estable con n=44 observaciones
- No tiene mecanismo de Attention nativo -> comparacion directa con GE

## Hipotesis
Si TCN supera a GE en MAE global pero se deteriora mas en shocks,
confirma que el mecanismo de Attention del LSTM es clave para volatilidad extrema.

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tcn import TCN
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    print(f'GPU: {gpus[0].name}')
else:
    print('CPU mode')

PROJECT_ROOT = Path('../..')
DATA_PATH    = PROJECT_ROOT / 'data/processed/master_dataset_fase2_multivariado.csv'
NLP_PATH     = PROJECT_ROOT / 'notebooks/fase2/output/01_nlp_sentimiento/sentimiento_mensual.csv'
GE_METRICAS  = PROJECT_ROOT / 'resultados/ge/ge_metricas.json'
OUT_DIR      = PROJECT_ROOT / 'resultados/tcn'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'TF: {tf.__version__}')
print(f'DATA_PATH ok: {DATA_PATH.exists()}')
print(f'NLP_PATH  ok: {NLP_PATH.exists()}')

In [ ]:
# Carga y agregacion por media provincial
df_raw = pd.read_csv(DATA_PATH, parse_dates=['fecha_evento'])
df = df_raw.groupby('fecha_evento').mean(numeric_only=True).reset_index()
df = df.sort_values('fecha_evento').reset_index(drop=True)
print(f'Agregado: {df.shape}')

# NLP con indices mejorados M1 y M2
df_nlp = pd.read_csv(NLP_PATH, encoding='utf-8-sig')
fc = [c for c in df_nlp.columns if any(k in c.lower() for k in ['fecha','periodo','mes','month'])][0]
df_nlp = df_nlp.rename(columns={fc: 'fecha_evento'})
df_nlp['fecha_evento']   = pd.to_datetime(df_nlp['fecha_evento'])
df_nlp['nlp_index']      = df_nlp['avg_sentiment'] * np.log1p(df_nlp['n_noticias_beto'])
df_nlp['nlp_index_lag1'] = df_nlp['nlp_index'].shift(1).fillna(0)

df = df.merge(df_nlp[['fecha_evento','nlp_index','nlp_index_lag1']], on='fecha_evento', how='left')
df['nlp_index']      = df['nlp_index'].fillna(0)
df['nlp_index_lag1'] = df['nlp_index_lag1'].fillna(0)
print(f'Con NLP: {df.shape}')

TARGET = 'produccion_t'
META   = ['fecha_evento', TARGET]
STRUCT = [c for c in df.columns if c not in META + ['nlp_index','nlp_index_lag1']]
NLP_F  = ['nlp_index', 'nlp_index_lag1']
ALL_F  = STRUCT + NLP_F
print(f'Features: {len(ALL_F)} ({len(STRUCT)} struct + {len(NLP_F)} NLP)')

In [ ]:
# Split 80/20 cronologico
TIMESTEPS = 6
n_total = len(df)
n_train = int(n_total * 0.80)
n_test  = n_total - n_train

df_train = df.iloc[:n_train].copy()
df_test  = df.iloc[n_train:].copy()

print(f'Train: {n_train} | {df_train["fecha_evento"].min().date()} -> {df_train["fecha_evento"].max().date()}')
print(f'Test:  {n_test}  | {df_test["fecha_evento"].min().date()} -> {df_test["fecha_evento"].max().date()}')

# Escalado sin data leakage
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_tr_raw = scaler_X.fit_transform(df_train[ALL_F])
X_te_raw = scaler_X.transform(df_test[ALL_F])
y_tr_sc  = scaler_y.fit_transform(df_train[[TARGET]])
y_te_sc  = scaler_y.transform(df_test[[TARGET]])

# PCA 95% (igual que GM v2 para comparabilidad)
from sklearn.decomposition import PCA
pca = PCA(n_components=0.95, random_state=SEED)
X_tr = pca.fit_transform(X_tr_raw)
X_te = pca.transform(X_te_raw)
print(f'PCA: {X_tr_raw.shape[1]} -> {pca.n_components_} componentes')

In [ ]:
# Construccion de tensores
def make_seq(X, y, ts):
    Xs, ys = [], []
    for i in range(ts, len(X)):
        Xs.append(X[i-ts:i])
        ys.append(y[i])
    return np.array(Xs), np.array(ys)

X_seq_tr, y_seq_tr = make_seq(X_tr, y_tr_sc, TIMESTEPS)
X_seq_te, y_seq_te = make_seq(X_te, y_te_sc, TIMESTEPS)

print(f'X_train: {X_seq_tr.shape}  <- (N, timesteps, features)')
print(f'X_test:  {X_seq_te.shape}')
print(f'Secuencias test: {n_test} - {TIMESTEPS} = {n_test - TIMESTEPS}')

In [ ]:
# Arquitectura TCN
# TCN usa convoluciones dilatadas: captura patrones en ventanas crecientes
# sin la recurrencia paso-a-paso del LSTM
# Dilations [1,2,4,8] significa que mira 1, 2, 4, 8 meses hacia atras

def build_tcn(input_shape, nb_filters=32, kernel_size=3, dilations=[1,2,4,8], dropout=0.2):
    inp = layers.Input(shape=input_shape, name='input')
    x = TCN(
        nb_filters=nb_filters,
        kernel_size=kernel_size,
        dilations=dilations,
        dropout_rate=dropout,
        return_sequences=False,
        activation='relu',
        name='tcn_layer'
    )(inp)
    x = layers.Dense(16, activation='relu', name='dense_16')(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(1, name='output')(x)
    return Model(inputs=inp, outputs=out, name='TCN_competidor')

input_shape = (X_seq_tr.shape[1], X_seq_tr.shape[2])
model_tcn = build_tcn(input_shape)
model_tcn.summary()
print(f'Input shape: {input_shape}')
print('Dilations [1,2,4,8]: ventanas de 1, 2, 4, 8 meses hacia atras')

In [ ]:
model_tcn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='mse',
    metrics=['mae']
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=15,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=7, min_lr=1e-6, verbose=1)
]

print('Entrenando TCN...')
history = model_tcn.fit(
    X_seq_tr, y_seq_tr,
    epochs=200,
    batch_size=8,
    validation_split=0.2,
    callbacks=callbacks,
    shuffle=False,
    verbose=1
)

In [ ]:
best_epoch    = int(np.argmin(history.history['val_loss'])) + 1
best_val_loss = float(min(history.history['val_loss']))

fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].plot(history.history['loss'],     label='Train', color='steelblue')
axes[0].plot(history.history['val_loss'], label='Val',   color='orange')
axes[0].set_title('Loss MSE')
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[1].plot(history.history['mae'],     label='Train', color='steelblue')
axes[1].plot(history.history['val_mae'], label='Val',   color='orange')
axes[1].set_title('MAE')
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.suptitle(f'TCN | best_epoch={best_epoch} | val_loss={best_val_loss:.4f}', fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'tcn_training_curves.png', dpi=150, bbox_inches='tight')
plt.close()
print(f'Best epoch={best_epoch} val_loss={best_val_loss:.4f}')

In [ ]:
# Evaluacion
y_pred_sc = model_tcn.predict(X_seq_te, verbose=0)
y_pred = scaler_y.inverse_transform(y_pred_sc).flatten()
y_true = scaler_y.inverse_transform(y_seq_te).flatten()

mae  = float(mean_absolute_error(y_true, y_pred))
rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
r2   = float(r2_score(y_true, y_pred))
smape= float(100*np.mean(2*np.abs(y_true-y_pred)/(np.abs(y_true)+np.abs(y_pred)+1e-8)))

# Naive MAE
y_all   = df['produccion_t'].values
y_t     = y_all[n_train:]
y_naive = y_all[n_train-1:-1]
naive_mae = mean_absolute_error(y_t, y_naive)
mase = mae / (naive_mae + 1e-8)

if GE_METRICAS.exists():
    with open(GE_METRICAS) as f:
        ge = json.load(f)
    ge_mae = ge.get('mae', ge.get('MAE', 0.0673))
else:
    ge_mae = 0.0673

print('=' * 65)
print('  COMPARATIVA FINAL — TODOS LOS MODELOS')
print('=' * 65)
print(f"  {'Modelo':<22} {'MAE':>8} {'RMSE':>8} {'R2':>8} {'MASE':>8}")
print('-' * 65)
rows = [
    ('Naive',         0.0161, None,   None,   1.00),
    ('XGBoost',       0.0471, 0.0542, -1.02,  2.60),
    ('TCN',           mae,    rmse,   r2,     mase),
    ('GM v2 NLP',     0.0646, 0.0771, -9.86,  4.02),
    ('GE sin NLP',    ge_mae, 0.0698, -2.34,  4.19),
    ('GM original',   0.0981, 0.1007, -5.96,  6.11),
    ('Prophet',       0.0919, 0.1051, -6.05,  5.72),
    ('SARIMA',        0.1006, 0.1179, -7.86,  6.26),
    ('SARIMAX+LSTM',  0.1969, 0.2926, -53.63, 12.26),
]
rows_sorted = sorted(rows, key=lambda x: x[1])
for nombre, m, r, r2_, ms in rows_sorted:
    r_str  = f'{r:.4f}' if r is not None else 'N/A'
    r2_str = f'{r2_:.2f}' if r2_ is not None else 'N/A'
    ms_str = f'{ms:.4f}'
    marca  = ' <- MEJOR' if m == rows_sorted[0][1] else ''
    print(f"  {nombre:<22} {m:>8.4f} {r_str:>8} {r2_str:>8} {ms_str:>8}{marca}")
print('=' * 65)

In [ ]:
# Analisis de shocks TCN
fechas_test = df['fecha_evento'].iloc[n_train + TIMESTEPS:].reset_index(drop=True)
df_test_eval = df.iloc[n_train + TIMESTEPS:].copy().reset_index(drop=True)
df_test_eval['variacion_pct'] = df_test_eval[TARGET].pct_change().abs() * 100
idx_shock = df_test_eval[df_test_eval['variacion_pct'] > 20].index.tolist()

print(f'Meses de shock en test TCN: {len(idx_shock)} / {len(y_true)}')

if len(idx_shock) > 0:
    mae_shock  = float(mean_absolute_error(y_true[idx_shock], y_pred[idx_shock]))
    deterioro  = (mae_shock - mae) / mae * 100
    print(f'MAE global:          {mae:.4f}')
    print(f'MAE shocks:          {mae_shock:.4f}')
    print(f'Deterioro en shocks: {deterioro:+.1f}%')
    print()
    print('Comparativa deterioro en shocks:')
    print(f'  GE sin NLP:  +2.3%   <- mas robusto')
    print(f'  XGBoost:     +16.5%')
    print(f'  TCN:         {deterioro:+.1f}%')
    print(f'  GM v2:       +29.2%')
    print(f'  Naive:       +27.9%')

In [ ]:
# Grafico predicciones vs real
fig, ax = plt.subplots(figsize=(12,5))
ax.plot(fechas_test, y_true, 'o-',  color='black',     lw=2, ms=5, label='Real')
ax.plot(fechas_test, y_pred, 's--', color='red',        lw=2, ms=5, label=f'TCN MAE={mae:.4f}')

xgb_path = PROJECT_ROOT / 'resultados/xgboost/xgb_predicciones.csv'
if xgb_path.exists():
    df_xgb = pd.read_csv(xgb_path)
    n_ov = min(len(fechas_test), len(df_xgb))
    ax.plot(fechas_test[:n_ov], df_xgb['pred_xgb'].values[:n_ov],
            '^:', color='darkorange', lw=1.5, ms=5, alpha=0.8, label='XGBoost 0.0471')

ge_path = PROJECT_ROOT / 'resultados/ge/ge_predicciones.csv'
if ge_path.exists():
    df_ge = pd.read_csv(ge_path)
    col_p = [c for c in df_ge.columns if 'pred' in c.lower()][0]
    n_ov  = min(len(fechas_test), len(df_ge))
    ax.plot(fechas_test[:n_ov], df_ge[col_p].values[:n_ov],
            'v:', color='steelblue', lw=1.5, ms=5, alpha=0.7, label='GE 0.0673')

ax.set_title('TCN vs XGBoost vs GE — Predicciones vs Real', fontweight='bold')
ax.set_xlabel('Fecha')
ax.set_ylabel('Produccion (media provincial)')
ax.legend()
ax.grid(alpha=0.3)
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(OUT_DIR / 'tcn_predicciones_vs_real.png', dpi=150, bbox_inches='tight')
plt.close()
print('Graficos guardados')

In [ ]:
# Guardar resultados
resultados = {
    'modelo': 'TCN_competidor',
    'MAE': mae, 'RMSE': rmse, 'R2': r2, 'sMAPE': smape, 'MASE': mase,
    'n_train': n_train, 'n_test': n_test,
    'best_val_loss': best_val_loss,
    'best_epoch': best_epoch,
    'timesteps': TIMESTEPS,
    'pca_components': int(pca.n_components_),
    'comparativa': {
        'Naive':       {'MAE': 0.0161, 'MASE': 1.00},
        'XGBoost':     {'MAE': 0.0471, 'MASE': 2.60},
        'TCN':         {'MAE': mae,    'MASE': mase},
        'GM_v2':       {'MAE': 0.0646, 'MASE': 4.02},
        'GE_sin_NLP':  {'MAE': ge_mae, 'MASE': 4.19},
        'GM_original': {'MAE': 0.0981, 'MASE': 6.11},
        'Prophet':     {'MAE': 0.0919, 'MASE': 5.72},
        'SARIMA':      {'MAE': 0.1006, 'MASE': 6.26},
        'SARIMAX_LSTM':{'MAE': 0.1969, 'MASE': 12.26},
    }
}
with open(OUT_DIR / 'tcn_metricas.json', 'w') as f:
    json.dump(resultados, f, indent=2)

pd.DataFrame({'fecha': fechas_test.values, 'real': y_true, 'pred_tcn': y_pred}).to_csv(
    OUT_DIR / 'tcn_predicciones.csv', index=False)

model_tcn.save(OUT_DIR / 'tcn_model.keras')

print('Archivos guardados en resultados/tcn/')
for f in sorted(OUT_DIR.iterdir()):
    print(f'  {f.name}')
print()
print('RESUMEN EJECUTIVO TCN')
print(f'TCN     MAE: {mae:.4f}')
print(f'XGBoost MAE: 0.0471')
print(f'GE      MAE: 0.0673')
print(f'GM v2   MAE: 0.0646')
if mae < 0.0471:
    print('RESULTADO: TCN supera a XGBoost -> red neuronal temporal gana')
elif mae < ge_mae:
    print('RESULTADO: TCN supera al GE pero no a XGBoost')
elif mae < 0.0646:
    print('RESULTADO: TCN entre GE y GM v2')
else:
    print('RESULTADO: LSTM-Attention supera a TCN -> arquitectura validada')